# 🧠 EgoRA Quickstart — Fine-Tune Without Forgetting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ArsSocratica/EgoRA/blob/main/notebooks/quickstart_colab.ipynb)
[![PyPI](https://img.shields.io/pypi/v/egora)](https://pypi.org/project/egora/)
[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.19410504.svg)](https://doi.org/10.5281/zenodo.19410504)

**EgoRA** (Entropy-Governed Orthogonality Regularization for Adaptation) is a dynamic regularization method for fine-tuning neural networks that prevents knowledge destruction.

This notebook shows three ways to use EgoRA, from simplest to most flexible:

1. **`EgoRATrainer`** — One-liner HuggingFace Trainer integration (recommended)
2. **`EgoRAPeftModel`** — PEFT-compatible wrapper
3. **Low-level API** — Full manual control

---

## 0. Install

In [ ]:
!pip install -q egora transformers datasets accelerate

In [ ]:
import egora
print(f"EgoRA version: {egora.__version__}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Prepare a dataset

We'll use a small instruction-tuning dataset for demonstration.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_NAME = "meta-llama/Llama-3.2-1B"  # Small model for demo

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load a small subset of Alpaca
dataset = load_dataset("tatsu-lab/alpaca", split="train[:500]")

def tokenize(example):
    prompt = f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"
    tokens = tokenizer(
        prompt, truncation=True, max_length=256, padding="max_length"
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset = dataset.map(tokenize, remove_columns=dataset.column_names)
dataset.set_format("torch")
print(f"Dataset: {len(dataset)} examples")

---

## 2. Option A: `EgoRATrainer` (Recommended)

The simplest way — a drop-in replacement for HuggingFace `Trainer`.

In [ ]:
from egora import EgoRATrainer, EgoRATrainingArguments, EgoRALoraConfig

config = EgoRALoraConfig(
    r=16,              # LoRA rank
    lora_alpha=32,     # LoRA scaling
    use_egora=True,    # Enable entropy-governed penalty
)

args = EgoRATrainingArguments(
    output_dir="./egora-quickstart",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    learning_rate=2e-4,
    logging_steps=10,
    bf16=torch.cuda.is_available(),
    save_strategy="no",  # Don't save checkpoints in demo
)

trainer = EgoRATrainer(
    model_name=MODEL_NAME,
    egora_config=config,
    args=args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    model_kwargs={"torch_dtype": torch.bfloat16} if torch.cuda.is_available() else {},
)

trainer.print_trainable_parameters()

In [ ]:
# Train!
trainer.train()

In [ ]:
# Post-training diagnostics — how much did the model rotate?
report = trainer.diagnose()
print(f"Mean rotation:   {report['theta_bar_deg']:.2f}°")
print(f"Damaged heads:   {report['damaged_fraction']*100:.1f}%")
print(f"Critical θ:      {report.get('theta_crit_deg', 'N/A')}°")

---

## 3. Option B: `EgoRAPeftModel` (PEFT-style API)

For users who prefer the PEFT workflow pattern.

In [ ]:
from transformers import AutoModelForCausalLM
from egora import EgoRAPeftModel, EgoRALoraConfig

config = EgoRALoraConfig(r=16, lora_alpha=32, use_egora=True)
model = EgoRAPeftModel.from_pretrained(MODEL_NAME, config)

model.print_trainable_parameters()

# One-liner loss in your training loop:
# total_loss = model.compute_total_loss(task_loss, logits)

# Save / load adapters:
# model.save_pretrained("./my-egora-adapter")
# model.load_adapter("./my-egora-adapter")

# Merge for deployment:
# base_model = model.merge_and_unload()

---

## 4. Option C: Low-Level API (Full Control)

For researchers who want full control over the training loop.

In [ ]:
from transformers import AutoModelForCausalLM
from egora import (
    apply_lora,
    get_total_egora_penalty,
    refresh_all_shadows,
    EntropyGovernor,
    compute_head_geometry,
)

# Load model & apply EgoRA adapters
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
lora_modules = apply_lora(base_model, rank=16, use_egora=True)
gov = EntropyGovernor(alpha=1.359, lam_floor=0.6931)

print(f"Adapters applied: {len(lora_modules)}")
print(f"Trainable params: {sum(p.numel() for p in base_model.parameters() if p.requires_grad):,}")

In [ ]:
# Example training step (pseudocode — plug in your own dataloader)
#
# optimizer = torch.optim.AdamW(
#     [p for p in base_model.parameters() if p.requires_grad], lr=2e-4
# )
#
# for step, batch in enumerate(dataloader):
#     outputs = base_model(**batch)
#     task_loss = outputs.loss
#
#     # Entropy-governed penalty
#     lam = gov.compute_lambda(outputs.logits)
#     penalty = get_total_egora_penalty(lora_modules)
#     total_loss = task_loss + lam * penalty
#
#     total_loss.backward()
#     optimizer.step()
#     optimizer.zero_grad()
#
#     # Refresh shadows every 100 steps
#     if step % 100 == 0:
#         refresh_all_shadows(lora_modules, momentum=0.9)

print("Low-level API ready — see comments above for training loop template.")

---

## 5. CLI Diagnostics (No Python Required)

EgoRA also has a command-line interface for quick analysis.

In [ ]:
# Show model info (layers, d_head, critical θ, LoRA param estimates)
!egora info meta-llama/Llama-3.2-1B

In [ ]:
# Compare base vs fine-tuned model (if you have a fine-tuned checkpoint):
# !egora diagnose meta-llama/Llama-3.2-1B ./my-finetuned-model -o results.json --plot

!egora version

---

## Resources

| | Link |
|---|---|
| 📦 **PyPI** | [`pip install egora`](https://pypi.org/project/egora/) |
| 💻 **GitHub** | [ArsSocratica/EgoRA](https://github.com/ArsSocratica/EgoRA) |
| 🤗 **Dataset** | [ArsSocratica/egora-benchmarks](https://huggingface.co/datasets/ArsSocratica/egora-benchmarks) |
| 🔖 **DOI** | [10.5281/zenodo.19410504](https://doi.org/10.5281/zenodo.19410504) |

### License

AGPL-3.0 with Academic Additional Permission. Free for research use (citation required).

**Contact**: mark@dillerop.com